# Tabela Silver — `ecommerce_enderecos`

Este notebook aplica regras de qualidade usando **PySpark**, separa registros válidos e rejeitados, e envia ambos para tabelas separadas no SQL Server.

A tabela Silver contém todos os registros, com colunas de auditoria:

- `invalidado`: `Sim` ou `Não`
- `is_valido`: `S` ou `N`
- `motivo_rejeicao`
- `data_hora_rejeicao`

Assim, registros inválidos não são descartados e também não é necessário criar uma tabela separada de rejeitados.

In [0]:
%run ../utils/utils.ipynb

## Imports e parâmetros

In [0]:



import uuid
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from functools import reduce
from datetime import datetime, timezone

# Variáveis do Processo
RUN_ID = str(uuid.uuid4())
TABELA_ALVO = "ecommerce_enderecos"
TABELA_DQ = "dq_monitoring_logs"

print(f"Iniciando processamento Silver - Endereços - Run ID: {RUN_ID}")


## Funções auxiliares


In [0]:
# 1. Carrega Bronze de Endereços
try:
    df_bronze_enderecos = ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)
except Exception as e:
    raise Exception(f"Erro: Tabela Bronze de {TABELA_ALVO} não encontrada.")

# 2. Isola o Micro-lote (Apenas endereços novos)
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver_atual = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    df_micro_lote = df_bronze_enderecos.join(df_silver_atual, "id_endereco", "left_anti")
else:
    df_micro_lote = df_bronze_enderecos

qtd_novos = df_micro_lote.count()
print(f"Registros novos no micro-lote de endereços: {qtd_novos}")

# 3. Referência de Clientes (Regra 2: O cliente deve existir)
def obter_referencia_ids(camada, tabela, coluna_id):
    if delta_existe(camada, tabela, STORAGE_OPTIONS):
        return ler_delta(camada, tabela, STORAGE_OPTIONS).select(coluna_id).dropDuplicates()
    else:
        schema = StructType([StructField(coluna_id, LongType(), True)])
        return spark.createDataFrame([], schema)

df_clientes_ref = obter_referencia_ids("bronze", "ecommerce_clientes", "id_cliente") \
    .withColumnRenamed("id_cliente", "id_cliente_ref")

print("Tabela de referência de Clientes carregada.")

## Anti-Join e Tabelas de Referência

In [0]:
# 1. Carrega Bronze de Rastreamento
try:
    df_bronze_rastreamento = ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)
except Exception as e:
    raise Exception(f"Erro: Tabela Bronze de {TABELA_ALVO} não encontrada.")

# 2. Isola o Micro-lote (Apenas rastreios novos)
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver_atual = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    df_micro_lote = df_bronze_enderecos.join(df_silver_atual, "id_endereco", "left_anti")
else:
    df_micro_lote = df_bronze_enderecos

qtd_novos = df_micro_lote.count()
print(f"Registros novos no micro-lote de rastreamento: {qtd_novos}")

# 3. Referência de Pedidos (AQUI É ONDE O df_pedidos_ref NASCE)
if delta_existe("bronze", "ecommerce_pedidos", STORAGE_OPTIONS):
    df_pedidos_ref = ler_delta("bronze", "ecommerce_pedidos", STORAGE_OPTIONS).select(
        F.col("id_pedido").alias("id_pedido_ref"),
        F.col("status_pedido").alias("status_pedido_ref")
    ).dropDuplicates(["id_pedido_ref"])
else:
    schema = StructType([
        StructField("id_pedido_ref", LongType(), True),
        StructField("status_pedido_ref", StringType(), True)
    ])
    df_pedidos_ref = spark.createDataFrame([], schema)

print("Tabela de referência de Pedidos carregada.")

In [0]:
print("Colunas disponíveis no df_micro_lote:", df_micro_lote.columns)

## Aplicação das 10 Regras de Qualidade

In [0]:
if qtd_novos > 0:
    # Parâmetros de Validação
    regex_cep = r"^\d{8}$"
    ufs_validas = ['AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MT', 'MS', 'MG', 'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN', 'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO']
    apelidos_validos = ['Casa', 'Trabalho', 'Outro']
    
    # Janelas Analíticas para Regras de Negócio
    w_id_endereco = Window.partitionBy("id_endereco")
    w_id_cliente = Window.partitionBy("id_cliente")

    # Prepara os dados (Limpeza e Agregação)
  # Prepara os dados (Limpeza e Agregação)
    df_base = df_micro_lote \
        .join(df_clientes_ref, df_micro_lote.id_cliente == df_clientes_ref.id_cliente_ref, "left") \
        .withColumn("cep_limpo", F.regexp_replace(F.col("cep").cast("string"), r"[^0-9]", "")) \
        .withColumn("lat_num", F.col("latitude").cast("double")) \
        .withColumn("lon_num", F.col("longitude").cast("double")) \
        .withColumn("qtd_id_endereco", F.count("*").over(w_id_endereco)) \
        .withColumn("qtd_enderecos_cliente", F.count("*").over(w_id_cliente)) \
        .withColumn("qtd_principal_cliente", F.sum(F.when(F.col("is_principal").cast("boolean") == True, 1).otherwise(0)).over(w_id_cliente))

    # Aplicação das 10 Regras de Endereços
    df_silver_enderecos = df_base \
        .withColumn("r1_id_endereco_falhou", F.col("id_endereco").isNull() | (F.col("id_endereco").cast("string") == "") | (F.col("qtd_id_endereco") > 1)) \
        .withColumn("r2_id_cliente_fk_falhou", F.col("id_cliente").isNull() | F.col("id_cliente_ref").isNull()) \
        .withColumn("r3_cep_falhou", F.col("cep_limpo").isNull() | (~F.col("cep_limpo").rlike(regex_cep))) \
        .withColumn("r4_estado_uf_falhou", F.col("estado").isNull() | (~F.col("estado").isin(ufs_validas))) \
        .withColumn("r5_coordenadas_br_falhou", F.col("lat_num").isNull() | F.col("lon_num").isNull() | (F.col("lat_num") < -33.75) | (F.col("lat_num") > 5.27) | (F.col("lon_num") < -73.99) | (F.col("lon_num") > -32.39)) \
        .withColumn("r6_unico_principal_falhou", F.col("qtd_principal_cliente") != 1) \
        .withColumn("r7_apelido_padrao_falhou", F.col("apelido").isNotNull() & (~F.col("apelido").isin(apelidos_validos))) \
        .withColumn("r8_limite_enderecos_falhou", F.col("qtd_enderecos_cliente") > 3) \
        .withColumn("r9_logradouro_falhou", F.col("logradouro").isNull() | (F.trim(F.col("logradouro")) == "")) \
        .withColumn("r10_numero_falhou", F.col("numero").isNull() | (F.trim(F.col("numero").cast("string")) == ""))

    print("Muralha de qualidade de endereços estruturada com sucesso.")
else:
    print("Etapa ignorada: não há micro-lote novo.")


## Catalogo de Regras e Logs

In [0]:
if qtd_novos > 0:
    catalogo_regras = [
        {"coluna": "r1_id_endereco_falhou", "regra": "R1_ID_ENDERECO_NULO_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r2_id_cliente_fk_falhou", "regra": "R2_CLIENTE_FK_ORFÃO", "severidade": "Critica"},
        {"coluna": "r3_cep_falhou", "regra": "R3_CEP_FORMATO_INVALIDO", "severidade": "Critica"},
        {"coluna": "r4_estado_uf_falhou", "regra": "R4_UF_INVALIDA", "severidade": "Critica"},
        {"coluna": "r5_coordenadas_br_falhou", "regra": "R5_COORDENADAS_FORA_DO_BRASIL", "severidade": "Aviso"},
        {"coluna": "r6_unico_principal_falhou", "regra": "R6_CLIENTE_SEM_ENDERECO_PRINCIPAL_UNICO", "severidade": "Critica"},
        {"coluna": "r7_apelido_padrao_falhou", "regra": "R7_APELIDO_FORA_DO_PADRAO", "severidade": "Aviso"},
        {"coluna": "r8_limite_enderecos_falhou", "regra": "R8_CLIENTE_COM_MAIS_DE_3_ENDERECOS", "severidade": "Aviso"},
        {"coluna": "r9_logradouro_falhou", "regra": "R9_LOGRADOURO_VAZIO", "severidade": "Critica"},
        {"coluna": "r10_numero_falhou", "regra": "R10_NUMERO_VAZIO", "severidade": "Critica"}
    ]

    total_registros = df_silver_enderecos.count()
    logs_list = []
    
    for r in catalogo_regras:
        qtd_falhas = df_silver_enderecos.filter(F.col(r["coluna"]) == True).count()
        if qtd_falhas > 0:
            logs_list.append((
                RUN_ID,
                TABELA_ALVO,
                r["regra"],
                "FAIL",
                r["severidade"],
                int(qtd_falhas),
                int(total_registros),
                datetime.now(timezone.utc),
                f"Bronze Delta ({TABELA_ALVO})"
            ))

    if logs_list:
        df_dq_monitoring_logs_novos = spark.createDataFrame(logs_list, schema=schema_dq_logs())
    else:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema=schema_dq_logs())

    # Extração de colunas com List Comprehensions (Corrigido para ler dicionários!)
    flags_criticas = [r["coluna"] for r in catalogo_regras if r["severidade"] == "Critica"]
    flags_avisos = [r["coluna"] for r in catalogo_regras if r["severidade"] == "Aviso"]

    # Reduce para criar as Flags Finais
    condicao_invalida_critica = reduce(lambda a, b: a | b, [F.col(c) for c in flags_criticas])
    condicao_aviso = reduce(lambda a, b: a | b, [F.col(c) for c in flags_avisos]) if flags_avisos else F.lit(False)

    df_silver_enderecos = df_silver_enderecos \
        .withColumn("silver_linha_valida", ~condicao_invalida_critica) \
        .withColumn("silver_tem_aviso", condicao_aviso) \
        .withColumn("silver_processed_at", F.current_timestamp()) \
        .withColumn("silver_run_id", F.lit(RUN_ID))

    print("Logs processados:", df_dq_monitoring_logs_novos.count())
    display(df_dq_monitoring_logs_novos)
else:
    print("Etapa ignorada: não há micro-lote novo.")

## Gravação (Silver e Logs Centralizados e Protegidos)

In [0]:
if qtd_novos > 0:
    # 1. Filtra as linhas boas e limpa colunas temporárias
    colunas_finais = df_micro_lote.columns + ["silver_processed_at", "silver_run_id", "silver_tem_aviso"]
    
    df_silver_validos = df_silver_enderecos \
        .filter(F.col("silver_linha_valida") == True) \
        .select(*colunas_finais)
        
    qtd_validos = df_silver_validos.count()
    print(f"Endereços aprovados para gravar na Silver: {qtd_validos}")

    if qtd_validos > 0:
        sucesso_silver = gravar_delta(
            df=df_silver_validos,
            camada="silver",
            tabela=TABELA_ALVO,
            storage_opts=STORAGE_OPTIONS,
            mode="append",
            particionar=False 
        )
        if sucesso_silver:
            print(f"Tabela Silver {TABELA_ALVO} atualizada!")

    # 2. Gravação dos Logs de Data Quality na RAIZ (Protegido contra repetição)
    if df_dq_monitoring_logs_novos.count() == 0:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())

    if delta_existe(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS):
        df_logs_historico = ler_delta(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS) \
            .filter(F.col("tabela") == TABELA_ALVO)
        
        condicao_join = [
            df_dq_monitoring_logs_novos.regra == df_logs_historico.regra,
            F.to_date(df_dq_monitoring_logs_novos.timestamp_execucao) == F.to_date(df_logs_historico.timestamp_execucao)
        ]
        
        df_logs_para_gravar = df_dq_monitoring_logs_novos.join(df_logs_historico, condicao_join, "left_anti")
    else:
        df_logs_para_gravar = df_dq_monitoring_logs_novos
        
    if df_logs_para_gravar.count() > 0 or not delta_existe(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS):
        sucesso_logs = gravar_delta(
            df=df_logs_para_gravar,
            camada="", # Vazio para salvar na raiz
            tabela=TABELA_DQ,
            storage_opts=STORAGE_OPTIONS,
            mode="append",
            particionar=False
        )
        if sucesso_logs:
            print("Logs de qualidade unificados processados na raiz com sucesso!")
else:
    print("Rotina finalizada sem alterações físicas.")

## Aplicar as 10 regras


In [0]:
# Aplicar as 10 regras de qualidade de ecommerce_enderecos
if qtd_validos > 0:
    w_id_endereco = Window.partitionBy("id_endereco_str")
    w_cliente = Window.partitionBy("id_cliente_str")

    ufs_validas = [
        "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG",
        "PA", "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"
    ]

    apelidos_validos = ["Casa", "Trabalho", "Outro"]

    df_contagens = (
        df_base
        .withColumn("cep_digits", F.regexp_replace(F.col("cep_str"), r"[^0-9]", "")) # LIMPEZA DO HÍFEN AQUI
        .withColumn("qtd_id_endereco_no_lote", F.count("*").over(w_id_endereco))
        .withColumn("qtd_enderecos_cliente_no_lote", F.count("*").over(w_cliente))
        .withColumn(
            "qtd_principal_cliente_no_lote",
            F.sum(F.when(F.col("is_principal_bool"), 1).otherwise(0)).over(w_cliente)
        )
    )

    df_silver_enderecos = (
        df_contagens
        .join(df_enderecos_ref, on="id_cliente_str", how="left")
        .withColumn("cliente_existe", F.coalesce(F.col("cliente_existe"), F.lit(False)))

        # R1: id_endereco não pode ser nulo nem duplicado.
        .withColumn("r1_id_endereco_falhou", F.col("id_endereco_str").isNull() | (F.col("id_endereco_str") == "") | (F.col("qtd_id_endereco_no_lote") > 1))

        # R2: id_cliente deve existir em ecommerce_clientes.
        .withColumn(
            "r2_id_cliente_fk_falhou",
            F.col("id_cliente_str").isNull()
            | (F.col("id_cliente_str") == "")
            | (~F.col("cliente_existe"))
        )

        # R3: cep deve ter exatamente 8 dígitos numéricos.
        .withColumn(
            "r3_cep_falhou",
            F.col("cep_digits").isNull()
            | (~F.col("cep_digits").rlike(r"^\d{8}$"))
        )

        # R4: estado deve ser UF válida.
        .withColumn(
            "r4_estado_falhou",
            F.col("estado_norm").isNull()
            | (~F.col("estado_norm").rlike(r"^[A-Z]{2}$"))
            | (~F.col("estado_norm").isin(ufs_validas))
        )

        # R5: latitude e longitude não podem ser nulas e devem estar no range do Brasil.
        .withColumn(
            "r5_lat_lon_falhou",
            F.col("latitude_num").isNull()
            | F.col("longitude_num").isNull()
            | (~F.col("latitude_num").between(-33.75, 5.27))
            | (~F.col("longitude_num").between(-73.99, -32.39))
        )

        # R6: cada cliente deve ter exatamente 1 endereço principal.
        .withColumn(
            "r6_endereco_principal_falhou",
            F.col("id_cliente_str").isNull()
            | (F.col("id_cliente_str") == "")
            | (F.col("qtd_principal_cliente_no_lote") != 1)
        )

        # R7: apelido deve ser Casa, Trabalho ou Outro.
        .withColumn(
            "r7_apelido_falhou",
            F.col("apelido_norm").isNull()
            | (~F.col("apelido_norm").isin(apelidos_validos))
        )

        # R8: nenhum cliente deve ter mais de 3 endereços cadastrados.
        .withColumn(
            "r8_max_3_enderecos_falhou",
            F.col("id_cliente_str").isNull()
            | (F.col("id_cliente_str") == "")
            | (F.col("qtd_enderecos_cliente_no_lote") > 3)
        )

        # R9: logradouro não pode ser nulo ou vazio.
        .withColumn(
            "r9_logradouro_falhou",
            F.col("logradouro_norm").isNull()
            | (F.col("logradouro_norm") == "")
        )

        # R10: numero não pode ser nulo.
        .withColumn(
            "r10_numero_falhou",
            F.col("numero").isNull()
            | F.col("numero_str").isNull()
            | (F.col("numero_str") == "")
        )
    )

    flags_regras = [
        "r1_id_endereco_falhou",
        "r2_id_cliente_fk_falhou",
        "r3_cep_falhou",
        "r4_estado_falhou",
        "r5_lat_lon_falhou",
        "r6_endereco_principal_falhou",
        "r7_apelido_falhou",
        "r8_max_3_enderecos_falhou",
        "r9_logradouro_falhou",
        "r10_numero_falhou",
    ]

    df_silver_enderecos = (
        df_silver_enderecos
        .withColumn(
            "silver_linha_valida",
            ~reduce(lambda a, b: a | b, [F.col(c) for c in flags_regras])
        )
    )

    display(df_silver_enderecos.limit(10))
else:
    print("Etapa ignorada: não há micro-lote novo.")


##  Resumo da validação


In [0]:
# Resumo das regras
if qtd_validos > 0:
    exprs = [
        F.sum(F.when(F.col(c), 1).otherwise(0)).alias(c)
        for c in flags_regras
    ]

    display(df_silver_enderecos.agg(*exprs))
    display(df_silver_enderecos.groupBy("silver_linha_valida").count())
else:
    print("Etapa ignorada: não há micro-lote novo.")



##  Gerar logs de DQ


In [0]:
# Gerar logs de DQ
if qtd_validos > 0:
    regras_dq = [
        {"coluna": "r1_id_endereco_falhou", "regra": "R1_ID_ENDERECO_NULO_OU_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r2_id_cliente_fk_falhou", "regra": "R2_ID_CLIENTE_INEXISTENTE", "severidade": "Critica"},
        {"coluna": "r3_cep_falhou", "regra": "R3_CEP_INVALIDO", "severidade": "Critica"},
        {"coluna": "r4_estado_falhou", "regra": "R4_UF_INVALIDA", "severidade": "Critica"},
        {"coluna": "r5_lat_lon_falhou", "regra": "R5_COORDENADAS_INVALIDAS", "severidade": "Critica"},
        {"coluna": "r6_endereco_principal_falhou", "regra": "R6_CLIENTE_SEM_EXATAMENTE_UM_ENDERECO_PRINCIPAL", "severidade": "Critica"},
        {"coluna": "r7_apelido_falhou", "regra": "R7_APELIDO_INVALIDO", "severidade": "Critica"},
        {"coluna": "r8_max_3_enderecos_falhou", "regra": "R8_CLIENTE_COM_MAIS_DE_3_ENDERECOS", "severidade": "Critica"},
        {"coluna": "r9_logradouro_falhou", "regra": "R9_LOGRADOURO_NULO_OU_VAZIO", "severidade": "Critica"},
        {"coluna": "r10_numero_falhou", "regra": "R10_NUMERO_NULO", "severidade": "Critica"},
    ]

    qtd_total = df_silver_enderecos.count()
    df_dq_monitoring_logs_novos = gerar_logs_por_regras(
        df_silver_enderecos,
        regras_dq,
        NOME_TABELA_DQ,
        qtd_total
    )

    print("Logs gerados:", df_dq_monitoring_logs_novos.count())
    display(df_dq_monitoring_logs_novos.limit(20))
else:
    print("Etapa ignorada: não há micro-lote novo.")


## Gravar Silver válida e logs

In [0]:
# Gravar Silver válida e logs
if qtd_validos > 0:
    # Silver contém somente registros válidos e colunas de negócio.
    colunas_silver_validas = [
        c for c in df_micro_lote.columns
        if c not in BRONZE_AUDIT_COLS
    ]

    df_silver_enderecos_validos = (
        df_silver_enderecos
        .filter(F.col("silver_linha_valida") == True)
        .select(*colunas_silver_validas)
        .withColumn("silver_processed_at", F.current_timestamp())
    )

    qtd_validos = df_silver_enderecos_validos.count()
    qtd_invalidos = df_silver_enderecos.filter(F.col("silver_linha_valida") == False).count()
    qtd_logs = df_dq_monitoring_logs_novos.count()

    print("Registros válidos para Silver:", qtd_validos)
    print("Registros inválidos enviados ao dq_monitoring_logs:", qtd_invalidos)
    print("Logs novos para dq_monitoring_logs:", qtd_logs)
    print("Colunas finais da Silver:", len(df_silver_enderecos_validos.columns))
    print(df_silver_enderecos_validos.columns)

    modo_silver = "overwrite" if (FORCAR_REPROCESSAMENTO and SOBRESCREVER_SILVER) else "append"
    print("Modo gravação Silver:", modo_silver)

    if qtd_validos > 0:
        gravar_delta_sem_particao(
            df=df_silver_enderecos_validos,
            caminho_delta=CAMINHO_SILVER_ENDERECOS,
            pasta_relativa=PASTA_SILVER_ENDERECOS if modo_silver == "overwrite" else None,
            mode=modo_silver
        )
        print("Silver gravada apenas com registros válidos.")
    else:
        print("Nenhum registro válido para gravar na Silver.")

    if qtd_logs > 0:
        if FORCAR_REPROCESSAMENTO and LIMPAR_LOGS_ANTIGOS_DA_TABELA:
            limpar_logs_antigos_da_tabela(NOME_TABELA_DQ)

        gravar_dq_logs(df_dq_monitoring_logs_novos)
        print("Logs gravados em squad1.dq_monitoring_logs.")
    else:
        print("Nenhum log novo para gravar.")

else:
    print("Nada para salvar: não há micro-lote novo.")


##  Gravar Silver e `dq_monitoring_logs`

In [0]:
#  confirmação da criação e salvamento
display(
    spark.table("squad1.dq_monitoring_logs")
    .orderBy(F.col("timestamp_execucao").desc())
)

##  Validação final e visualização das tabelas

In [0]:
print("===== VALIDAÇÃO FINAL =====")

# 1. Validação da tabela Silver principal
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_validacao_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    print(f"Registros na Silver {TABELA_ALVO}:", df_validacao_silver.count())
    display(df_validacao_silver.limit(20))
else:
    print(f"A tabela Silver {TABELA_ALVO} ainda não existe no Data Lake.")

# 2. Validação da tabela de Logs de Qualidade (na raiz do Data Lake)
if delta_existe(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS):
    df_logs_validacao = ler_delta(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS)
    
    # Filtra para mostrar apenas os logs referentes à tabela de endereços
    df_logs_filtrados = df_logs_validacao.filter(F.col("tabela") == TABELA_ALVO)
    
    print(f"Logs na {TABELA_DQ} para {TABELA_ALVO}:", df_logs_filtrados.count())
    display(df_logs_filtrados.orderBy(F.col("timestamp_execucao").desc()).limit(20))
else:
    print(f"Tabela {TABELA_DQ} ainda não existe no Data Lake.")